# 01 · Load & Exploratory Data Analysis

**What you'll learn**
- How to load a whitespace-delimited flat file into pandas with named columns.
- How to describe a run-to-failure dataset (engines, cycles, per-engine length).
- How to spot **dead sensors** (constant values → no information).
- How to identify the **6 operating regimes** hiding inside the operational
  settings — and why that matters for feature engineering.
- Which sensors show a **monotonic drift toward failure** (these are the
  ones we'll build features on).

Every finding here motivates a decision we make in notebook 02.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# Add repo root to sys.path so `from src.data import ...` works when the
# notebook is opened from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

## 1 · Load the training file

The loader is a thin wrapper around `pd.read_csv` that names the columns for
us. Look at `src/data.py` — it's about 30 lines of code and worth reading
before we call it.

In [ ]:
from src.data import load_fd004, load_rul_fd004, add_rul_train, SENSOR_COLS, OP_COND_COLS

df = load_fd004("train")
print(f"shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

### What are the columns?

| Column | Meaning |
|--------|---------|
| `unit` | Engine ID (1 … 249 for training) |
| `cycle` | Operating cycle since the engine was installed. Row 1 = brand new, higher cycle = older. |
| `os1`, `os2`, `os3` | Operational settings: altitude, Mach number, throttle-resolver angle. Values change from cycle to cycle as the engine flies different missions. |
| `s1` … `s21` | 21 sensor channels — temperatures, pressures, fan speeds, fuel flow, etc. This is what we'll monitor. |

There is no "label" column — no cycle is pre-marked as anomalous. We'll
derive labels in notebook 05 using the RUL (remaining useful life).

## 2 · How many engines, how long do they run?

In [ ]:
n_engines = df["unit"].nunique()
cycles_per_engine = df.groupby("unit")["cycle"].max()

print(f"# engines: {n_engines}")
print(f"cycles per engine — min={cycles_per_engine.min()}, "
      f"median={int(cycles_per_engine.median())}, max={cycles_per_engine.max()}")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(cycles_per_engine, bins=40, color="#4c78a8", edgecolor="white")
ax.set_xlabel("Cycles until failure")
ax.set_ylabel("Number of engines")
ax.set_title("How long each training engine survived")
plt.show()

**Reading the plot.** Engines fail across a wide range — the shortest
lives out around 150 cycles, the longest past 500. That variability is real
(different fault mixes, different operating profiles), and it means we can't
just say "trigger the alarm at cycle 200" for every engine. The threshold
needs to be about the *state* of the sensors, not the age of the engine.

## 3 · Append the true RUL to each row

For training data we know when each engine failed (its last recorded cycle).
That means for any earlier cycle we can compute the *remaining useful life*
as `max_cycle - current_cycle`. We'll need this later for evaluation.

In [ ]:
df = add_rul_train(df)
df[["unit", "cycle", "rul"]].head()

In [ ]:
# Sanity check: the last row of every engine should have RUL = 0.
last = df.sort_values("cycle").groupby("unit").tail(1)
assert (last["rul"] == 0).all(), "RUL not zero at failure — loader bug?"
print("✓ RUL is 0 at the failure cycle for every engine")

## 4 · One engine's trajectory

In [ ]:
def plot_engine_sensors(df, unit, sensors, ncols=3):
    sub = df[df["unit"] == unit].sort_values("cycle")
    n = len(sensors)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 2.2), sharex=True)
    for ax, s in zip(np.array(axes).flatten(), sensors):
        ax.plot(sub["cycle"], sub[s], color="#4c78a8", linewidth=0.9)
        ax.set_title(s, fontsize=9)
    for ax in np.array(axes).flatten()[len(sensors):]:
        ax.set_visible(False)
    fig.suptitle(f"Engine {unit} · raw sensor trajectories", y=1.02)
    fig.tight_layout()
    plt.show()

plot_engine_sensors(df, unit=1, sensors=SENSOR_COLS)

**Reading the plot.** Two things jump out:

1. **Some sensors are flat lines** — s1, s5, s6, s10, s16, s18, s19 are
   constants (or effectively so) for this engine. These are dead channels —
   they carry no information about degradation. We'll drop them.

2. **Some sensors have a huge cycle-to-cycle jitter** even though the
   engine is nowhere near failure. That jitter is real — it comes from the
   engine flying different operating conditions from one cycle to the next.
   Which brings us to…

## 5 · The 6 operating regimes

Look at the three `os*` columns for engine 1:

In [ ]:
sub = df[df["unit"] == 1].sort_values("cycle")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.scatter(sub["os1"], sub["os2"], c=sub["os3"], cmap="viridis", s=14, alpha=0.7)
ax.set_xlabel("os1  (altitude proxy)")
ax.set_ylabel("os2  (Mach proxy)")
ax.set_title("Engine 1 · operating settings — clear clusters of flight conditions")
plt.colorbar(ax.collections[0], label="os3")
plt.show()

The scatter isn't a cloud — it's a handful of discrete clusters. The
same engine flies a handful of *distinct* operating profiles (e.g. cruise at
high altitude vs takeoff at sea level), and it visits them repeatedly over
its life. Let's confirm there are exactly 6 across the whole training set.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=6, n_init=10, random_state=0)
regime = km.fit_predict(df[OP_COND_COLS].values)
df = df.assign(regime=regime)
print("Rows per regime:")
print(df["regime"].value_counts().sort_index())

**Why this matters.** Look at what happens to sensor s2 (a temperature)
if we don't split by regime:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
axes[0].hist(df["s2"], bins=60, color="#8c8c8c", edgecolor="white")
axes[0].set_title("s2 across all cycles (regimes mixed) — looks bimodal")
axes[0].set_xlabel("s2")

for r, color in zip(range(6), sns.color_palette("tab10", 6)):
    axes[1].hist(df.loc[df["regime"] == r, "s2"], bins=40, color=color, alpha=0.55,
                 label=f"regime {r}", edgecolor="white", linewidth=0.5)
axes[1].set_title("s2 split by regime — each regime is tight and unimodal")
axes[1].set_xlabel("s2")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

The "bimodal" look on the left is not a real bimodality — it's *six*
regimes stacked on top of each other. If we fed the raw sensor value to an
anomaly detector, it would happily flag every cruise cycle as different from
every takeoff cycle. We don't want that. In notebook 02 we'll normalise each
sensor **within its regime**, and the detectors will then focus on the actual
degradation, not the flight condition.

**Rule of thumb learned here:** always plot per-group histograms if you have
a categorical setting variable — mixed populations look like fake anomalies.

## 6 · Which sensors carry a degradation signal?

We want features that separate *healthy* from *near-failure*. The
easiest way to spot which raw sensors carry a signal at all: overlay their
mean trajectory for early-life cycles vs late-life cycles, per regime.

In [ ]:
# Split each engine's life into "early" (first 30%) and "late" (last 30%).
def life_bin(g):
    max_c = g["cycle"].max()
    return np.where(g["cycle"] < 0.3 * max_c, "early",
           np.where(g["cycle"] > 0.7 * max_c, "late", "mid"))

df["life"] = df.groupby("unit", group_keys=False).apply(
    lambda g: pd.Series(life_bin(g), index=g.index)
)

# Mean sensor value in each (regime, life-bin) cell, focused on regime 0.
r0 = df[df["regime"] == 0]
summary = r0.groupby("life")[SENSOR_COLS].mean().loc[["early", "late"]].T
summary["abs_shift"] = (summary["late"] - summary["early"]).abs()
summary["norm_shift"] = summary["abs_shift"] / summary["early"].abs()
summary.sort_values("norm_shift", ascending=False).head(15).style.format("{:.4f}")

The top of that table is our shortlist of sensors that carry a
degradation signal in regime 0 (they shift measurably from early life to late
life). In the next notebook we'll build features from *these* sensors, not
all 21.

Let's visualise one that shifts a lot (usually s4 or s11) alongside one
that doesn't (s1 or s5):

In [ ]:
def overlay_healthy_vs_degraded(df, regime, sensor):
    r = df[df["regime"] == regime]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(r.loc[r["life"] == "early", sensor], bins=40,
            color="#4c78a8", alpha=0.65, label="early life", edgecolor="white", linewidth=0.5)
    ax.hist(r.loc[r["life"] == "late",  sensor], bins=40,
            color="#e45756", alpha=0.65, label="late life",  edgecolor="white", linewidth=0.5)
    ax.set_title(f"{sensor}  (regime {regime}) — early vs late life")
    ax.set_xlabel(sensor); ax.legend()
    plt.show()

overlay_healthy_vs_degraded(df, regime=0, sensor="s4")
overlay_healthy_vs_degraded(df, regime=0, sensor="s1")

**Reading the plots.** For s4, the late-life distribution has shifted
noticeably away from the early-life distribution — that's the degradation
signal we want to catch. For s1, the two distributions overlap almost
perfectly — this sensor is not useful and we'll drop it in the next
notebook.

## 7 · Correlations between sensors

Highly correlated sensors carry redundant information. Not a problem for
Isolation Forest (trees don't care about multicollinearity), but useful to
know when we're choosing which sensors to base features on — no point
engineering rolling means for both s7 and s8 if they're 99% correlated.

In [ ]:
# Correlation on regime 0 only so regime effects don't dominate.
alive = [s for s in SENSOR_COLS if df[s].nunique() > 1]
corr = df.loc[df["regime"] == 0, alive].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink": 0.7},
            annot=False, xticklabels=True, yticklabels=True)
ax.set_title("Sensor correlations · regime 0 only")
plt.show()

## 8 · Sanity-check the test set

Nothing to model here yet — just confirm the loader works and the RUL file
lines up with the test engines.

In [ ]:
test = load_fd004("test")
rul_test = load_rul_fd004()
print(f"test shape: {test.shape}")
print(f"# test engines: {test['unit'].nunique()}")
print(f"# RUL entries: {len(rul_test)}")
assert len(rul_test) == test["unit"].nunique()
print("✓ one RUL value per test engine")

## Takeaways for notebook 02

The next notebook (`02_feature_engineering`) will act on what we learned here:

1. **Drop dead sensors** (identified by zero-variance / near-zero-variance).
2. **Normalise every sensor within its operating regime**, so regime
   changes don't masquerade as anomalies.
3. **Focus feature engineering on the sensors that showed measurable early
   vs late shift** (top of the `norm_shift` table).
4. For each engineered feature, we'll state the **hypothesis** the feature
   captures, the **math**, and a **validation plot** showing it separates
   healthy from near-failure cycles.